# Demo 1: Parallel Computing Practice

**Objective:** Experience the power of parallel computing by analyzing patient data with `multiprocessing.Pool`, observe real speedup, and create your first SLURM script.

## Step 1: Local Parallel Analysis (10 minutes)

### Create Sample Patient Data Files

In [ ]:
import pandas as pd
import numpy as np
import os

# DEBUG MODE: Set to True for faster testing
DEBUG_MODE = True  # Change to False for full analysis

# Configure based on mode
if DEBUG_MODE:
    n_files = 4          # Only 4 files for quick testing
    min_patients = 100   # Smaller cohorts
    max_patients = 500
    print("🐣 DEBUG MODE: Creating baby-sized dataset for quick testing...")
else:
    n_files = 20         # Full 20 files
    min_patients = 1000  # Full-size cohorts
    max_patients = 5000
    print("🚀 FULL MODE: Creating complete dataset...")

# Create directory for patient data
os.makedirs('patient_cohorts', exist_ok=True)

# Generate sample patient cohort files
for i in range(n_files):
    n_patients = np.random.randint(min_patients, max_patients)
    
    # Simulate patient data
    patients = pd.DataFrame({
        'patient_id': range(n_patients),
        'age': np.random.normal(65, 15, n_patients),
        'risk_score': np.random.beta(2, 5, n_patients),  # Most patients low risk
        'hospital_days': np.random.poisson(3, n_patients),
        'comorbidities': np.random.poisson(2, n_patients)
    })
    
    patients.to_csv(f'patient_cohorts/cohort_{i:02d}.csv', index=False)
    
print(f"Created {n_files} patient cohort files!")

### Run Parallel Analysis

> 🚨 **Important:** Jupyter notebooks have limitations with `multiprocessing.Pool`. Use the notebook-friendly version below, or save as a `.py` file for full multiprocessing!

In [ ]:
import pandas as pd
import time
from concurrent.futures import ProcessPoolExecutor
import os

# DEBUG MODE: Global variable accessible to worker processes
DEBUG_MODE = True  # Change to False for full analysis

def analyze_patient_cohort(patient_file):
    """Analyze a single patient cohort file"""
    # Load patient data
    df = pd.read_csv(patient_file)
    
    # Simulate complex analysis - shorter sleep in debug mode
    sleep_time = 0.01 if DEBUG_MODE else 0.1
    time.sleep(sleep_time)  # Represents actual computation time
    
    # Return summary statistics
    return {
        'file': patient_file,
        'n_patients': len(df),
        'avg_age': df['age'].mean(),
        'high_risk_count': (df['risk_score'] > 0.8).sum(),
        'completion_time': time.time()
    }

# Sequential vs Parallel comparison
n_files = 4 if DEBUG_MODE else 20
patient_files = [f'patient_cohorts/cohort_{i:02d}.csv' for i in range(n_files)]

print(f"📊 Analyzing {len(patient_files)} cohort files...")

# Sequential processing
print("⏳ Running sequential analysis...")
start_time = time.time()
sequential_results = [analyze_patient_cohort(f) for f in patient_files]
sequential_time = time.time() - start_time

# Parallel processing (notebook-friendly version)
print("⚡ Running parallel analysis...")
start_time = time.time()

# Try parallel processing - fall back to sequential if it fails in notebook
try:
    if os.getenv('JUPYTER_RUNNING') or 'ipykernel' in globals():
        # In Jupyter - use threading instead of multiprocessing
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=4) as executor:
            parallel_results = list(executor.map(analyze_patient_cohort, patient_files))
        print("🔧 Used ThreadPoolExecutor (notebook-friendly)")
    else:
        # In script - use multiprocessing
        with ProcessPoolExecutor(max_workers=4) as executor:
            parallel_results = list(executor.map(analyze_patient_cohort, patient_files))
        print("🚀 Used ProcessPoolExecutor (full parallelism)")
except Exception as e:
    print(f"⚠️ Parallel processing failed ({e}), using sequential...")
    parallel_results = sequential_results
    
parallel_time = time.time() - start_time

print(f"\n📈 RESULTS:")
print(f"Sequential: {sequential_time:.2f}s")
print(f"Parallel:   {parallel_time:.2f}s")
print(f"Speedup:    {sequential_time/parallel_time:.2f}x")

### Alternative: Pure multiprocessing.Pool (for standalone scripts)

In [ ]:
import multiprocessing as mp
import pandas as pd
import time

# DEBUG MODE: Global variable accessible to worker processes
DEBUG_MODE = True  # Change to False for full analysis

def analyze_patient_cohort(patient_file):
    """Analyze a single patient cohort file"""
    # Load patient data
    df = pd.read_csv(patient_file)
    
    # Simulate complex analysis - shorter sleep in debug mode
    sleep_time = 0.01 if DEBUG_MODE else 0.1
    time.sleep(sleep_time)  # Represents actual computation time
    
    # Return summary statistics
    return {
        'file': patient_file,
        'n_patients': len(df),
        'avg_age': df['age'].mean(),
        'high_risk_count': (df['risk_score'] > 0.8).sum(),
        'completion_time': time.time()
    }

if __name__ == '__main__':  # REQUIRED for multiprocessing.Pool!
    # Sequential vs Parallel comparison
    n_files = 4 if DEBUG_MODE else 20
    patient_files = [f'patient_cohorts/cohort_{i:02d}.csv' for i in range(n_files)]

    print(f"📊 Analyzing {len(patient_files)} cohort files...")

    # Sequential processing
    start_time = time.time()
    sequential_results = [analyze_patient_cohort(f) for f in patient_files]
    sequential_time = time.time() - start_time

    # Parallel processing - ONLY works in standalone .py files
    start_time = time.time()
    with mp.Pool(processes=4) as pool:
        parallel_results = pool.map(analyze_patient_cohort, patient_files)
    parallel_time = time.time() - start_time

    print(f"Sequential: {sequential_time:.2f}s")
    print(f"Parallel: {parallel_time:.2f}s")
    print(f"Speedup: {sequential_time/parallel_time:.2f}x")

### Test Different Process Counts

In [ ]:
# Test different numbers of processes (notebook-friendly version)
max_processes = 4 if DEBUG_MODE else 8  # Fewer processes in debug mode
process_counts = [1, 2, 4] if DEBUG_MODE else [1, 2, 4, 8]

print("\n🧪 Testing different process counts...")
for n_processes in process_counts:
    start_time = time.time()
    
    try:
        if os.getenv('JUPYTER_RUNNING') or 'ipykernel' in globals():
            # Use threading in notebooks
            from concurrent.futures import ThreadPoolExecutor
            with ThreadPoolExecutor(max_workers=n_processes) as executor:
                results = list(executor.map(analyze_patient_cohort, patient_files))
        else:
            # Use multiprocessing in scripts
            with ProcessPoolExecutor(max_workers=n_processes) as executor:
                results = list(executor.map(analyze_patient_cohort, patient_files))
    except Exception:
        # Fallback to sequential
        results = [analyze_patient_cohort(f) for f in patient_files]
        
    elapsed = time.time() - start_time
    print(f"{n_processes} workers: {elapsed:.2f}s")

## Step 2: SLURM Script Creation (5 minutes)

Create `my_health_analysis.sh`:

In [ ]:
%%bash
#!/bin/bash
#SBATCH --job-name=my_first_health_analysis
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=1:00:00
#SBATCH --output=logs/health_analysis_%j.out

echo "Starting health data analysis on $(hostname)"
echo "Number of CPUs: $SLURM_CPUS_PER_TASK"
echo "Memory allocated: $SLURM_MEM_PER_NODE MB"

# Load modules (if on Wynton)
# module load python/3.9

# Run your parallel analysis
python parallel_patient_analysis.py
echo "Analysis complete!"

## Success Criteria

✅ **Speedup Observed:** Parallel version runs 2-4x faster than sequential
✅ **Understanding Gained:** Can explain when to use processes vs threads
✅ **SLURM Ready:** Created a working SLURM script for cluster submission
✅ **Resource Awareness:** Understand relationship between data size and memory requirements

> 💡 **Debug Tip:** Start with `DEBUG_MODE = True` to quickly test your setup, then switch to `DEBUG_MODE = False` to see real performance differences with larger datasets.

## Common Issues & Solutions

- **"No speedup observed":** Check if analysis function is CPU-bound. Add more computation if needed.
- **"Runs slower with more processes":** Overhead domination. Try smaller datasets or more computation per task.
- **"Memory errors":** Reduce sample data size or increase memory allocation in SLURM script.
- **"Can't get attribute 'analyze_patient_cohort'":** Jupyter notebook limitation! Use the `concurrent.futures` version above or save as `.py` file.
- **"RuntimeError about bootstrapping":** You forgot the `if __name__ == '__main__':` guard! This is required on macOS/Windows.
- **"NameError: name 'DEBUG_MODE' is not defined":** Make sure DEBUG_MODE is defined at the global level, not inside the main block.

## 💡 Pro Tips

- **In Jupyter:** Use `ThreadPoolExecutor` or `ProcessPoolExecutor` from `concurrent.futures`
- **In Scripts:** Use `multiprocessing.Pool` for maximum performance
- **For I/O tasks:** Threading often works just as well as multiprocessing
- **For CPU tasks:** True multiprocessing gives the best speedup